##**Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Silver
### Objetivo
Processar os itens de pedido da Bronze com o pipeline de qualidade mais completo do Squad 2, aplicando validação de schema, deduplicação, integridade referencial e consistência de preço.
### Origem e Destino
| Item | Valor |
| **Origem** | `squad2/bronze/ecommerce_itens_pedido` (Delta Lake) |
| **Destino** | `squad2/silver/ecommerce_itens_pedido` (Delta Lake) |
| **Controle** | `silver/control/ecommerce_itens_pedido.json` |
| **Sala de Espera** | `squad2/silver/waiting_room/ecommerce_itens_pedido` |
| **Quarentena** | `squad2/silver/quarantine/ecommerce_itens_pedido_preco_inconsistente` |
| **Modo de escrita** | `append` incremental por arquivo de origem |
### Regras de Qualidade Aplicadas
| # | Regra | Campo(s) | Critério | Ação quando falha |
| 1 | Validação de Schema | 6 colunas obrigatórias | Todas presentes no lote | **Bloqueia o lote inteiro** |
| 2 | Integridade Referencial (FK) | `id_pedido` | Deve existir na Silver de Pedidos | Linha vai para **Sala de Espera** |
| 3 | Quantidade positiva | `quantidade` | Maior que `0` | Linha descartada |
| 4 | Consistência de preço | `preco_unitario` | >= 40% do `preco_lista` do catálogo | Linha vai para **Quarentena** |
| 5 | Deduplicação | `id_item_pedido` | Único dentro do lote e na Silver existente | Duplicata descartada |
### Colunas Obrigatórias (Regra 1)
 `id_item_pedido`, `id_pedido`, `sku`, `quantidade`, `preco_unitario`, `desconto_aplicado`
### Coluna de Auditoria Adicionada
| Coluna | Descrição |
| `silver_processed_at` | Timestamp de processamento na Silver |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | Conexão ADLS, `get_storage_options`, `get_squad2_client` |
| `silver/ecommerce_pedidos` | Fonte de IDs válidos de pedido (Regra 2) |
| `silver/ecommerce_produtos` ou `bronze/ecommerce_produtos` | Catálogo de preços (Regra 4) |


In [0]:
%run ../utils/feat_squad2_99_helpers

- Configuração de Caminhos
Define os caminhos ABFSS para todas as tabelas envolvidas no processamento:
- Origem (Bronze de itens)
- Destino (Silver de itens)
- Dependências (Silver/Bronze de pedidos e produtos)
- Áreas de governança (Sala de Espera e Quarentena)

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_itens_pedido"

# Caminhos Delta oficiais
path_bronze = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/{TABELA}"
path_silver = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"
path_control = f"silver/control/{TABELA}.json"

# Caminhos de dependências para as Regras 2 e 4
path_silver_pedidos  = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ecommerce_pedidos"
path_silver_produtos = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/ecommerce_produtos"
path_bronze_produtos = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/bronze/ecommerce_produtos"

# Pastas de Governança (Sala de Espera e Quarentena de Preço)
path_waiting_room = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/waiting_room/{TABELA}"
path_quarantine_price = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/quarantine/{TABELA}_preco_inconsistente"

- Leitura da Bronze e Controle Incremental

In [0]:
try:
    # 1. Abre a tabela Bronze de Itens de Pedido
    dt_bronze = DeltaTable(path_bronze, storage_options=get_storage_options())
    df_pandas = dt_bronze.to_pandas()
    
    # 2. CONTROLE INCREMENTAL: Baixa a lista de arquivos já processados
    squad2_client = get_squad2_client()
    file_client = squad2_client.get_file_client(path_control)
    processados = set()
    
    if file_client.exists():
        conteudo = file_client.download_file().readall().decode('utf-8')
        processados = set(json.loads(conteudo))
    
    # Filtra apenas os dados inéditos trazidos pela Bronze
    df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
    
    if df_novos_dados.empty:
        print(" Camada Silver de Itens de Pedido em dia! Nenhum dado novo.")
    else:
        print(f" Micro-lote detectado: {len(df_novos_dados)} linhas brutas para processar.")
        
        # -------------------------------------------------------------------------
        # REGRA 1: Validação de Schema Completo presente no lote
        # -------------------------------------------------------------------------
        colunas_obrigatorias = ['id_item_pedido', 'id_pedido', 'sku', 'quantidade', 'preco_unitario', 'desconto_aplicado']
        colunas_faltantes = [col for col in colunas_obrigatorias if col not in df_novos_dados.columns]
        
        if colunas_faltantes:
            raise ValueError(f" BLOQUEIO DE LOTE: Colunas obrigatórias ausentes no arquivo: {colunas_faltantes}")
        print("   Regra 1: Schema completo validado.")

        # -------------------------------------------------------------------------
        # REGRA 5 (Parte 1): Deduplicação interna dentro do próprio micro-lote
        # -------------------------------------------------------------------------
        df_working = df_novos_dados.drop_duplicates(subset=['id_item_pedido'], keep='last').copy()
        
        # REGRA 5 (Parte 2): Deduplicação externa contra a tabela Silver existente (Evita duplicar receita)
        if DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
            dt_silver = DeltaTable(path_silver, storage_options=get_storage_options())
            ids_existentes = set(dt_silver.to_pandas()['id_item_pedido'].unique())
            df_working = df_working[~df_working['id_item_pedido'].isin(ids_existentes)].copy()
        print("   Regra 5: Deduplicação aplicada com sucesso.")

        # -------------------------------------------------------------------------
        # REGRA 3: quantidade deve ser > 0
        # -------------------------------------------------------------------------
        df_working = df_working[df_working['quantidade'] > 0].copy()
        print("   Regra 3: Linhas com quantidade zerada ou negativa descartadas.")

        # -------------------------------------------------------------------------
        # REGRA 2: Integridade de id_pedido (FK na Silver de Pedidos)
        # -------------------------------------------------------------------------
        if DeltaTable.is_deltatable(path_silver_pedidos, storage_options=get_storage_options()):
            dt_pedidos = DeltaTable(path_silver_pedidos, storage_options=get_storage_options())
            pedidos_validos = set(dt_pedidos.to_pandas()['id_pedido'].unique())
            
            # Divide entre registros com pai localizado e órfãos (vão para a Sala de Espera)
            df_com_pai = df_working[df_working['id_pedido'].isin(pedidos_validos)].copy()
            df_sala_espera = df_working[~df_working['id_pedido'].isin(pedidos_validos)].copy()
        else:
            # Fallback seguro: como a Silver de pedidos ainda vai ser criada, isolamos para a Sala de Espera
            print("   Nota Regra 2: Tabela Silver de Pedidos não inicializada. Dados direcionados para a Sala de Espera.")
            df_com_pai = pd.DataFrame(columns=df_working.columns)
            df_sala_espera = df_working.copy()

        # Grava os órfãos na tabela/pasta de espera se houver algum
        if not df_sala_espera.empty:
            write_deltalake(path_waiting_room, df_sala_espera, mode="append", storage_options=get_storage_options())
            print(f"   {len(df_sala_espera)} itens isolados na Sala de Espera (Aguardando processamento da Silver de Pedidos).")

        # -------------------------------------------------------------------------
        # REGRA 4: preço_unitario > 0 e consistente com o catálogo (Silver/Bronze de Produtos)
        # -------------------------------------------------------------------------
        if not df_com_pai.empty:
            # Decide qual catálogo ler (Usa Silver se existir, senão usa a Bronze que já tem dados)
            path_catalogo = path_silver_produtos if DeltaTable.is_deltatable(path_silver_produtos, storage_options=get_storage_options()) else path_bronze_produtos
            dt_produtos = DeltaTable(path_catalogo, storage_options=get_storage_options())
            df_produtos_ref = dt_produtos.to_pandas()[['sku', 'preco_lista']].drop_duplicates(subset=['sku'])
            
            # Faz o cruzamento de preço (Lookup)
            df_validacao_preco = df_com_pai.merge(df_produtos_ref, on='sku', how='left')
            
            # Condição: preço > 0 e não pode ser mais que 60% menor que o preço de lista (limite de 40% do valor)
            condicao_preco_valido = (df_validacao_preco['preco_unitario'] > 0) & \
                                    (df_validacao_preco['preco_lista'].notna()) & \
                                    (df_validacao_preco['preco_unitario'] >= df_validacao_preco['preco_lista'] * 0.4)
            
            df_silver_final = df_validacao_preco[condicao_preco_valido].drop(columns=['preco_lista']).copy()
            df_erro_preco = df_validacao_preco[~condicao_preco_valido].drop(columns=['preco_lista']).copy()
            
            # Se houver erros de preço (fraude ou erro de digitação), isola na quarentena
            if not df_erro_preco.empty:
                write_deltalake(path_quarantine_price, df_erro_preco, mode="append", storage_options=get_storage_options())
                print(f"   Alerta Regra 4: {len(df_erro_preco)} linhas bloqueadas por inconsistência de preço (>60% menor que catálogo).")
        else:
            df_silver_final = pd.DataFrame(columns=df_working.columns)

        # -------------------------------------------------------------------------
        # GRAVAÇÃO FINAL NA CAMADA SILVER & ATUALIZAÇÃO DO CONTROLE
        # -------------------------------------------------------------------------
        if not df_silver_final.empty:
            # Aplica a coluna de auditoria obrigatória
            df_silver_final['silver_processed_at'] = datetime.now()
            
            # Limpa formatos de data para compatibilidade Delta
            for col in df_silver_final.columns:
                if pd.api.types.is_datetime64_any_dtype(df_silver_final[col]):
                    df_silver_final[col] = df_silver_final[col].dt.tz_localize(None)
            
            # Escrita definitiva na Silver
            write_deltalake(path_silver, df_silver_final, mode="append", storage_options=get_storage_options())
            print(f"   Gravação concluída: {len(df_silver_final)} linhas salvas em {path_silver}.")
        else:
            print("   Nenhuma linha elegível para gravação imediata na Silver nesta rodada.")

        # Atualiza o arquivo JSON de controle na pasta de governança
        arquivos_processados_nesta_rodada = set(df_novos_dados['bronze_source_file'].unique())
        todos_processados = list(processados.union(arquivos_processados_nesta_rodada))
        file_client.upload_data(json.dumps(todos_processados), overwrite=True)
        print("   Arquivo de controle JSON atualizado com sucesso em silver/control/.")

except Exception as e:
    print(f" Erro crítico no pipeline da Silver: {str(e)}")
    raise